# PCOS Prediction — 01: Data Loading & Cleaning

**Dataset:** PCOS data without infertility (Kaggle, collected from 10 hospitals, Kerala)  
**Goal:** Predict `PCOS (Y/N)` (binary) from clinical, hormonal, ultrasound and lifestyle features.

This is notebook **1 of 4** in the PCOS prediction pipeline:

| # | Notebook | What it does |
|---|---|---|
| 1 | `01_Data_Loading_and_Cleaning.ipynb` | *(this notebook)* Load raw data, find and fix every data quality issue |
| 2 | `02_Exploratory_Data_Analysis.ipynb` | Univariate/bivariate EDA, statistical tests, correlation analysis |
| 3 | `03_Preprocessing_and_Feature_Engineering.ipynb` | Feature selection, engineering, train/test split, scaling |
| 4 | `04_Model_Building_and_Evaluation.ipynb` | Logistic Regression, Random Forest, XGBoost — tuned and compared |

## This notebook's structure
| Section | What it does |
|---|---|
| 1 | Setup & helper functions |
| 2 | Load raw data & initial profile |
| 3 | Systematic data quality assessment — find every issue *before* touching anything |
| 4 | Data cleaning — fix exactly what Section 3 revealed |
| 5 | Post-cleaning validation |

**Output:** `data/pcos_cleaned.csv`, consumed by every downstream notebook.

## Section 1 — Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, chi2_contingency, pointbiserialr
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 60)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100


In [2]:
# Helper function — Cramér's V
# Measures association between two categorical variables (0 = no association, 1 = perfect).
# Used later instead of Pearson correlation for categorical features vs the binary target.
#
# scipy.stats.contingency.association() computes Cramér's V directly from a contingency
# table in a single call — no manual chi2 arithmetic needed.
# Note: scipy's version does not apply the Bergsma-Zeijlstra small-sample bias correction,
# but with n=541 the difference from the corrected version is < 0.01 and negligible.
from scipy.stats.contingency import association

def cramers_v(x, y):
    mask = x.notna() & y.notna()
    ct = pd.crosstab(x[mask], y[mask])
    return association(ct, method='cramer')


## Section 2 — Load Raw Data & Initial Profile
We start from the original Excel file as downloaded from Kaggle.  
The first thing to do is check how many sheets exist — loading without specifying a sheet  
silently gives the wrong one.


In [3]:
xl = pd.ExcelFile('../data/PCOS_data_without_infertility.xlsx')
print("Sheets in this file:", xl.sheet_names)


Sheets in this file: ['Instructions', 'Full_new']


Two sheets are present: `Instructions` (just documentation — 20 rows, 4 columns)  
and `Full_new` (the actual data). We must load `Full_new` explicitly.  
Loading without `sheet_name='Full_new'` silently gives you the instructions sheet

In [4]:
df_raw = xl.parse('Full_new')
print(f"Shape: {df_raw.shape}")
df_raw.head()


Shape: (541, 45)


,Sl. No,Patient File No.,PCOS (Y/N),Age (yrs),Weight (Kg),Height(Cm),BMI,Blood Group,Pulse rate(bpm),RR (breaths/min),Hb(g/dl),Cycle(R/I),Cycle length(days),Marraige Status (Yrs),Pregnant(Y/N),No. of aborptions,I beta-HCG(mIU/mL),II beta-HCG(mIU/mL),FSH(mIU/mL),LH(mIU/mL),FSH/LH,Hip(inch),Waist(inch),Waist:Hip Ratio,TSH (mIU/L),AMH(ng/mL),PRL(ng/mL),Vit D3 (ng/mL),PRG(ng/mL),RBS(mg/dl),Weight gain(Y/N),hair growth(Y/N),Skin darkening (Y/N),Hair loss(Y/N),Pimples(Y/N),Fast food (Y/N),Reg.Exercise(Y/N),BP _Systolic (mmHg),BP _Diastolic (mmHg),Follicle No. (L),Follicle No. (R),Avg. F size (L) (mm),Avg. F size (R) (mm),Endometrium (mm),Unnamed: 44
0,1,1,0,28,44.6,152.0,19.300000,15,78,22,10.48,2,5,7.0,0,0,1.99,1.99,7.95,3.68,2.160326,36,30,0.833333,0.68,2.07,45.16,17.1,0.57,92.0,0,0,0,0,0,1.0,0,110,80,3,3,18.0,18.0,8.5,NaN
1,2,2,0,36,65.0,161.5,24.921163,15,74,20,11.70,2,5,11.0,1,0,60.80,1.99,6.73,1.09,6.174312,38,32,0.842105,3.16,1.53,20.09,61.3,0.97,92.0,0,0,0,0,0,0.0,0,120,70,3,5,15.0,14.0,3.7,NaN
2,3,3,1,33,68.8,165.0,25.270891,11,72,18,11.80,2,5,10.0,1,0,494.08,494.08,5.54,0.88,6.295455,40,36,0.900000,2.54,6.63,10.52,49.7,0.36,84.0,0,0,0,1,1,1.0,0,120,80,13,15,18.0,20.0,10.0,NaN
3,4,4,0,37,65.0,148.0,29.674945,13,72,20,12.00,2,5,4.0,0,0,1.99,1.99,8.06,2.36,3.415254,42,36,0.857143,16.41,1.22,36.90,33.4,0.36,76.0,0,0,0,0,0,0.0,0,120,70,2,2,15.0,14.0,7.5,NaN
4,5,5,0,25,52.0,161.0,20.060954,11,72,18,10.00,2,5,1.0,1,0,801.45,801.45,3.98,0.90,4.422222,37,30,0.810811,3.57,2.26,30.09,43.8,0.38,84.0,0,0,0,1,0,0.0,0,120,80,3,4,16.0,14.0,7.0,NaN


## Section 3 — Systematic Data Quality Assessment

We go through five checks in order:
1. Column name formatting  
2. Data types — which columns are object when they should be numeric  
3. Bad values inside those object columns  
4. Missing values  
5. Invalid codes in categorical columns  
6. Clinically impossible values  


### 3.1 — Column name formatting

In [5]:
# Check for leading/trailing whitespace in column names
# This is invisible to the eye but breaks exact-name lookups: df['BMI '] != df['BMI']
import re

raw_names = df_raw.columns.tolist()
has_whitespace = [c for c in raw_names if c != re.sub(r'\s+', ' ', c).strip()]
print(f"Columns with whitespace issues ({len(has_whitespace)}):")
for c in has_whitespace:
    print(f"  repr: {repr(c)}")


Columns with whitespace issues (5):
  repr: ' Age (yrs)'
  repr: 'Height(Cm) '
  repr: 'Pulse rate(bpm) '
  repr: '  I   beta-HCG(mIU/mL)'
  repr: 'II    beta-HCG(mIU/mL)'


In [6]:
# Fix: standardise column names
df = df_raw.copy()
df.columns = [re.sub(r'\s+', ' ', c).strip() for c in df.columns]
print("Column names after cleanup:")
print(df.columns.tolist())


Column names after cleanup:
['Sl. No', 'Patient File No.', 'PCOS (Y/N)', 'Age (yrs)', 'Weight (Kg)', 'Height(Cm)', 'BMI', 'Blood Group', 'Pulse rate(bpm)', 'RR (breaths/min)', 'Hb(g/dl)', 'Cycle(R/I)', 'Cycle length(days)', 'Marraige Status (Yrs)', 'Pregnant(Y/N)', 'No. of aborptions', 'I beta-HCG(mIU/mL)', 'II beta-HCG(mIU/mL)', 'FSH(mIU/mL)', 'LH(mIU/mL)', 'FSH/LH', 'Hip(inch)', 'Waist(inch)', 'Waist:Hip Ratio', 'TSH (mIU/L)', 'AMH(ng/mL)', 'PRL(ng/mL)', 'Vit D3 (ng/mL)', 'PRG(ng/mL)', 'RBS(mg/dl)', 'Weight gain(Y/N)', 'hair growth(Y/N)', 'Skin darkening (Y/N)', 'Hair loss(Y/N)', 'Pimples(Y/N)', 'Fast food (Y/N)', 'Reg.Exercise(Y/N)', 'BP _Systolic (mmHg)', 'BP _Diastolic (mmHg)', 'Follicle No. (L)', 'Follicle No. (R)', 'Avg. F size (L) (mm)', 'Avg. F size (R) (mm)', 'Endometrium (mm)', 'Unnamed: 44']


### 3.2 — Data types: which columns are object when they should not be?

In [7]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 541 entries, 0 to 540
Data columns (total 45 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Sl. No                 541 non-null    int64  
 1   Patient File No.       541 non-null    int64  
 2   PCOS (Y/N)             541 non-null    int64  
 3   Age (yrs)              541 non-null    int64  
 4   Weight (Kg)            541 non-null    float64
 5   Height(Cm)             541 non-null    float64
 6   BMI                    541 non-null    float64
 7   Blood Group            541 non-null    int64  
 8   Pulse rate(bpm)        541 non-null    int64  
 9   RR (breaths/min)       541 non-null    int64  
 10  Hb(g/dl)               541 non-null    float64
 11  Cycle(R/I)             541 non-null    int64  
 12  Cycle length(days)     541 non-null    int64  
 13  Marraige Status (Yrs)  540 non-null    float64
 14  Pregnant(Y/N)          541 non-null    int64  
 15  No. of aborptions

In [8]:
# Isolate: columns that are object dtype but should be numeric based on their names
# (hormone levels, measurements — not PCOS Y/N which is legitimately binary)
object_cols = df.select_dtypes(include='object').columns.tolist()
print("Object-dtype columns:", object_cols)
print()
print("Expected: only measurement columns should show up here.")
print("If a measurement column is object, it means at least one row has a non-numeric value")
print("that forced pandas to read the whole column as text.")


Object-dtype columns: ['II beta-HCG(mIU/mL)', 'AMH(ng/mL)', 'Unnamed: 44']

Expected: only measurement columns should show up here.
If a measurement column is object, it means at least one row has a non-numeric value
that forced pandas to read the whole column as text.


### 3.3 — Find the exact bad values in those columns

In [9]:
# For each object-dtype column, show precisely which rows fail numeric conversion.
# We do NOT convert yet — we just identify.

def find_non_numeric_rows(df, col):
    temp = pd.to_numeric(df[col], errors='coerce')
    bad_mask = temp.isna() & df[col].notna()  # was not already NaN, but failed conversion
    return df.loc[bad_mask, ['Sl. No', col]]

for col in object_cols:
    bad = find_non_numeric_rows(df, col)
    if len(bad) > 0:
        print(f"Column '{col}': {len(bad)} non-numeric value(s) found")
        print(bad.to_string(index=False))
        print()


Column 'II beta-HCG(mIU/mL)': 1 non-numeric value(s) found
 Sl. No II beta-HCG(mIU/mL)
    124               1.99.

Column 'AMH(ng/mL)': 1 non-numeric value(s) found
 Sl. No AMH(ng/mL)
    306          a

Column 'Unnamed: 44': 1 non-numeric value(s) found
 Sl. No Unnamed: 44
    181           .



**What we found:**  
- `AMH(ng/mL)` row 306: value `'a'` — a stray letter, clearly a data entry error.  
- `II beta-HCG(mIU/mL)` row 124: value `'1.99.'` — a double decimal point typo.  

Both of these need to be fixed before doing any numerical analysis on these columns.  
The EDA in the previous version ran on these columns while they were still object dtype —  
which is why every cell that called `.mean()` or `.quantile()` on them threw a `TypeError`.


### 3.4 — Missing values

In [10]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
print(missing_df)


                       Missing Count  Missing %
Unnamed: 44                      539      99.63
Marraige Status (Yrs)              1       0.18
Fast food (Y/N)                    1       0.18


All missing counts are ≤ 1 out of 541 rows (< 0.2%). This is negligibly small.  
**Decision:** leave these as `NaN` rather than imputing. Imputing 1 missing value out of 541  
has essentially zero effect on any statistic, and inventing data we don't have is riskier  
than leaving a tiny gap. Statistical functions used in EDA handle `NaN` by excluding the  
row from that specific calculation, so no computation will silently break.


### 3.5 — Invalid codes in categorical columns

In [11]:
# Cycle(R/I): should only be 2 (Regular) or 4 (Irregular) — those are the only valid codes.
# Check if any other value appears.
print("Cycle(R/I) unique values:", sorted(df['Cycle(R/I)'].unique()))
print()
print("Value counts:")
print(df['Cycle(R/I)'].value_counts().sort_index())


Cycle(R/I) unique values: [np.int64(2), np.int64(4), np.int64(5)]

Value counts:
Cycle(R/I)
2    390
4    150
5      1
Name: count, dtype: int64


In [12]:
# Show the row(s) with the invalid value
invalid_cycle = df[~df['Cycle(R/I)'].isin([2, 4])]
print(f"Rows with invalid Cycle(R/I) code: {len(invalid_cycle)}")
print(invalid_cycle[['Sl. No', 'Cycle(R/I)', 'PCOS (Y/N)']])


Rows with invalid Cycle(R/I) code: 1
     Sl. No  Cycle(R/I)  PCOS (Y/N)
512     513           5           1


Row 513 has `Cycle(R/I) = 5`, which is not a valid code. We cannot know whether the  
patient had a regular (2) or irregular (4) cycle, so this must be set to `NaN` rather  
than guessing. Guessing would silently introduce bias.


In [13]:
# Blood Group — check what codes appear (should be category labels, not a continuous measurement)
print("Blood Group unique codes:", sorted(df['Blood Group'].unique()))
print()
print("These are arbitrary numeric codes for blood types (e.g. 11=A+, 12=A-, 13=B+, etc.)")
print("They carry NO arithmetic meaning. Computing Pearson correlation with these codes")
print("is statistically incorrect — the difference between code 11 and 17 doesn't mean 'more blood group'.")


Blood Group unique codes: [np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]

These are arbitrary numeric codes for blood types (e.g. 11=A+, 12=A-, 13=B+, etc.)
They carry NO arithmetic meaning. Computing Pearson correlation with these codes
is statistically incorrect — the difference between code 11 and 17 doesn't mean 'more blood group'.


### 3.6 — Clinically impossible values

In [14]:
# Blood pressure: check for values outside the range compatible with life.
# Normal range: Systolic 80-200 mmHg, Diastolic 40-130 mmHg
# Values outside this almost certainly indicate data entry errors, not real measurements.

bp_cols = ['BP _Systolic (mmHg)', 'BP _Diastolic (mmHg)']
print(df[bp_cols].describe().round(1))


       BP _Systolic (mmHg)  BP _Diastolic (mmHg)
count                541.0                 541.0
mean                 114.7                  76.9
std                    7.4                   5.6
min                   12.0                   8.0
25%                  110.0                  70.0
50%                  110.0                  80.0
75%                  120.0                  80.0
max                  140.0                 100.0


In [15]:
# Find rows with values that cannot be physiologically real
impossible_bp = df[
    (df['BP _Systolic (mmHg)'] < 70) |
    (df['BP _Diastolic (mmHg)'] < 35)
]
print(f"Rows with impossible blood pressure values: {len(impossible_bp)}")
print(impossible_bp[['Sl. No'] + bp_cols + ['PCOS (Y/N)']])


Rows with impossible blood pressure values: 2
     Sl. No  BP _Systolic (mmHg)  BP _Diastolic (mmHg)  PCOS (Y/N)
161     162                   12                    80           0
200     201                  120                     8           0


Two rows with impossible values:  
- Systolic = **12** mmHg is incompatible with life (would be circulatory arrest). The other BP  
  value in the same row is 80 mmHg (normal diastolic), so this is almost certainly a missed digit:  
  12 → **120**. Cross-check: 120/80 is a perfectly normal blood pressure reading.  
- Diastolic = **8** mmHg is also impossible. The systolic in that row is 120 mmHg (normal),  
  so again a missed digit: 8 → **80**. Cross-check: 120/80 confirms the correction.


### 3.6b — Clinically impossible values, continued: Pulse rate
Running the same impossible-value check we just used for blood pressure across the other vitals — not just the ones that looked suspicious at a glance.

In [16]:
# Pulse rate: check for values outside the range compatible with life.
# Normal resting pulse: roughly 60-100 bpm. Even severe bradycardia rarely goes below 40.
# We check the same way we checked blood pressure above -- same logic, same threshold style.

print(df['Pulse rate(bpm)'].describe())
print()
impossible_pulse = df[df['Pulse rate(bpm)'] < 40]
print(f"Rows with impossible pulse rate: {len(impossible_pulse)}")
print(impossible_pulse[['Sl. No', 'Pulse rate(bpm)', 'PCOS (Y/N)']])


count    541.000000
mean      73.247689
std        4.430285
min       13.000000
25%       72.000000
50%       72.000000
75%       74.000000
max       82.000000
Name: Pulse rate(bpm), dtype: float64

Rows with impossible pulse rate: 2
     Sl. No  Pulse rate(bpm)  PCOS (Y/N)
223     224               18           0
296     297               13           0


Two more impossible values, missed on the first pass through this section:  
- Sl. No 224: Pulse rate = **13** bpm — incompatible with life.  
- Sl. No 297: Pulse rate = **18** bpm — also incompatible with life.  

Unlike the blood pressure fix above, there's no adjacent normal-looking value in the same row  
to cross-check a "missing digit" story against (13→130 and 18→180 are both implausible pulse  
rates too), so we can't confidently reconstruct the intended value the way we could for BP.  
**Decision:** set both to `NaN` rather than guess — the same rule already used for the invalid  
`Cycle(R/I)` code in 3.5.

### 3.7 — Identifier and empty columns to drop

In [17]:
# Unnamed: 44 — check what's in it
print("Unnamed: 44 non-null count:", df['Unnamed: 44'].notna().sum(), "out of", len(df))
print()
print("'Sl. No' and 'Patient File No.' are row identifiers, not clinical features.")
print("Including them in any analysis would be meaningless.")
print()
print("Summary of all columns to drop:")
print("  'Sl. No'          — row index")
print("  'Patient File No.'— patient identifier")
print("  'Unnamed: 44'     — empty trailing column (2 non-null out of 541)")


Unnamed: 44 non-null count: 2 out of 541

'Sl. No' and 'Patient File No.' are row identifiers, not clinical features.
Including them in any analysis would be meaningless.

Summary of all columns to drop:
  'Sl. No'          — row index
  'Patient File No.'— patient identifier
  'Unnamed: 44'     — empty trailing column (2 non-null out of 541)


### Summary of all issues found in Section 3

| Issue | Column | Details | Fix |
|---|---|---|---|
| Whitespace in names | multiple | Invisible spaces in headers | Strip & compress |
| Non-numeric value | AMH(ng/mL) | Row 306: value `'a'` | Set to NaN |
| Typo in numeric | II beta-HCG | Row 124: `'1.99.'` (double decimal) | Strip trailing dot → numeric |
| Invalid category code | Cycle(R/I) | Row 513: value `5` (only 2 or 4 are valid) | Set to NaN |
| Impossible value | BP Systolic | Row 162: 12 mmHg (incompatible with life) | Correct to 120 |
| Impossible value | BP Diastolic | Row 201: 8 mmHg (incompatible with life) | Correct to 80 |
| Impossible value | Pulse rate(bpm) | Rows 224, 297: 18 and 13 bpm (incompatible with life) | Set to NaN (no reliable correction) |
| Junk/identifier cols | Sl. No, Patient File No., Unnamed: 44 | Not features | Drop |

We found **all of these through code**, not by assuming them upfront.


## Section 4 — Data Cleaning
Applying exactly the fixes identified in Section 3 — nothing more, nothing less.


In [18]:
df_clean = df.copy()

# Fix 1: AMH — set the stray letter to NaN, then convert column to float
df_clean['AMH(ng/mL)'] = pd.to_numeric(df_clean['AMH(ng/mL)'], errors='coerce')

# Fix 2: II beta-HCG — strip the trailing dot typo, then convert to float
df_clean['II beta-HCG(mIU/mL)'] = (
    df_clean['II beta-HCG(mIU/mL)']
    .astype(str)
    .str.rstrip('.')
    .pipe(pd.to_numeric, errors='coerce')
)

# Fix 3: Cycle(R/I) — invalid code 5 → NaN
df_clean.loc[~df_clean['Cycle(R/I)'].isin([2, 4]), 'Cycle(R/I)'] = np.nan

# Fix 4: Blood pressure impossible values — clear missing-digit story, so we correct rather than null
df_clean.loc[df_clean['BP _Systolic (mmHg)'] == 12, 'BP _Systolic (mmHg)'] = 120
df_clean.loc[df_clean['BP _Diastolic (mmHg)'] == 8,  'BP _Diastolic (mmHg)'] = 80

# Fix 5: Pulse rate impossible values — no reliable correction story, so set to NaN
df_clean.loc[df_clean['Pulse rate(bpm)'] < 40, 'Pulse rate(bpm)'] = np.nan

# Fix 6: Drop identifier and empty columns
df_clean = df_clean.drop(columns=['Sl. No', 'Patient File No.', 'Unnamed: 44'])

print("Cleaning complete.")
print(f"Shape after cleaning: {df_clean.shape}")


Cleaning complete.
Shape after cleaning: (541, 42)


## Section 5 — Post-Cleaning Validation
Verify every fix actually worked. Never assume a fix applied correctly — check it.


In [19]:
print("=== Dtypes of previously-object columns ===")
print(df_clean[['AMH(ng/mL)', 'II beta-HCG(mIU/mL)']].dtypes)

print("\n=== Cycle(R/I) value counts after fix ===")
print(df_clean['Cycle(R/I)'].value_counts(dropna=False))

print("\n=== BP values after fix ===")
print(sorted(df_clean['BP _Systolic (mmHg)'].unique()))
print(sorted(df_clean['BP _Diastolic (mmHg)'].unique()))

print("\n=== Pulse rate after fix ===")
print(df_clean['Pulse rate(bpm)'].describe())

print("\n=== Missing values in final cleaned data ===")
missing_final = df_clean.isnull().sum()
missing_final = missing_final[missing_final > 0]
print(missing_final)

print("\n=== Final shape ===")
print(df_clean.shape)


=== Dtypes of previously-object columns ===
AMH(ng/mL)             float64
II beta-HCG(mIU/mL)    float64
dtype: object

=== Cycle(R/I) value counts after fix ===
Cycle(R/I)
2.0    390
4.0    150
NaN      1
Name: count, dtype: int64

=== BP values after fix ===
[np.int64(100), np.int64(110), np.int64(120), np.int64(130), np.int64(140)]
[np.int64(60), np.int64(70), np.int64(80), np.int64(100)]

=== Pulse rate after fix ===
count    539.000000
mean      73.461967
std        2.689637
min       70.000000
25%       72.000000
50%       72.000000
75%       74.000000
max       82.000000
Name: Pulse rate(bpm), dtype: float64

=== Missing values in final cleaned data ===
Pulse rate(bpm)          2
Cycle(R/I)               1
Marraige Status (Yrs)    1
AMH(ng/mL)               1
Fast food (Y/N)          1
dtype: int64

=== Final shape ===
(541, 42)


In [20]:
from pathlib import Path
print(Path.cwd())

/home/harshitbhatt/Projects/PCOS project/notebooks


In [21]:
import os
os.makedirs("../data", exist_ok=True)

df_clean.to_csv("../data/pcos_cleaned.csv", index=False)
print("Saved as data/pcos_cleaned.csv")
print(f"Shape: {df_clean.shape}")

Saved as data/pcos_cleaned.csv
Shape: (541, 42)
